# M2-01: Практическая реализация декомпозиции генерации документов

## Контекст

Этот notebook демонстрирует экспериментальную реализацию подхода к генерации образовательных документов в проекте [LearnFlow AI](https://github.com/bbaron/learnflow-ai).

Реализация независимо пришла к тем же выводам, что описаны в статье ["Navigating the Path of Writing"](https://www.arxiv.org/abs/2404.13919) - декомпозиция процесса генерации на этапы планирования структуры и генерации контента значительно улучшает качество и управляемость результата.

## Проблема

В LearnFlow AI при генерации образовательных материалов столкнулись с типичными проблемами монолитной генерации:

- **Непредсказуемость результата** - LLM генерирует весь материал за один вызов без промежуточного контроля
- **Дорогие итерации** - если результат не устраивает, приходится перегенерировать весь документ целиком
- **Сложность интеграции источников** - неясно как эффективно учитывать внешние материалы (конспекты студентов)

### Текущая архитектура LearnFlow (проблемная)

```mermaid
graph TB
    subgraph "Двойная генерация и потеря контроля"
        A[input_processing] --> B[generating_content]
        B --> C[recognition_handwritten]
        C --> D[synthesis_material]
        D --> E[edit_material]
    end

    style B fill:#ff9999
    style D fill:#ff9999

    B -.->|"⚠️ Генерация #1<br/>без учета конспектов"| B1[Материал v1]
    D -.->|"⚠️ Генерация #2<br/>переписывание"| D1[Материал v2]
```

## Решение: Двухэтапная генерация

Разделение процесса на два этапа:
1. **Планирование структуры** - генерация иерархической структуры документа
2. **Генерация контента** - заполнение структуры содержанием

### Целевая архитектура с декомпозицией

```mermaid
graph TB
    subgraph "Декомпозированный подход"
        A[input_processing] --> B[recognition_handwritten]
        B --> C[planning_structure]
        C --> D{HITL Review}
        D -->|"📝 Уточнить"| C
        D -->|"✅ Утвердить"| E[parallel_section_generation]
        E --> F[document_assembly]
    end

    style C fill:#99ff99
    style D fill:#ffff99
    style E fill:#99ff99

    C -.->|"💡 Структура<br/>~200 токенов"| C1[План]
    E -.->|"📄 Контент<br/>по секциям"| E1[Текст]
```

## Преимущество для HITL

Декомпозиция естественным образом создает точку для Human-in-the-Loop взаимодействия:
- ✅ **Дешевая валидация** - структура это всего ~100-200 токенов vs 2000+ для полного текста
- ✅ **Быстрая итерация** - перегенерация структуры занимает секунды
- ✅ **Контроль качества** - пользователь видит "план" до генерации и может его скорректировать

Хотя HITL не является центральным паттерном в LearnFlow AI, именно декомпозиция делает его эффективным.

---

## Что демонстрирует notebook

1. **Структурированный вывод** - Pydantic модели для формализации структуры документа
2. **Конфигурация персонализации** - настройка генерации под ML/intermediate уровень
3. **Двухфазная генерация** - отдельные промпты для планирования и HITL-итераций
4. **Интеграция с Telegram** - полноценный bot с FSM для демонстрации workflow
5. **Автоматизация решений** - определение намерений пользователя через structured output

## 1. Настройка окружения

## 1. Setup and Environment

In [1]:
import os
from pathlib import Path
from typing import List, Literal
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from jinja2 import Template
from dotenv import load_dotenv

# Загружаем переменные окружения
project_root = Path().cwd().parent.parent
env_local = project_root / ".env.local"
env_file = project_root / ".env"

if env_local.exists():
    load_dotenv(env_local)
    print(f"✓ Loaded .env.local from {env_local}")
elif env_file.exists():
    load_dotenv(env_file)
    print(f"✓ Loaded .env from {env_file}")
else:
    print("⚠ No .env file found")

# Проверяем обязательные API ключи
openai_api_key = os.getenv("OPENAI_API_KEY")
telegram_token = os.getenv("TELEGRAM_TOKEN")

if not openai_api_key:
    raise ValueError("OPENAI_API_KEY not found in environment")
if not telegram_token:
    raise ValueError("TELEGRAM_TOKEN not found in environment")

print(f"✓ OpenAI API key loaded: {openai_api_key[:8]}...")
print(f"✓ Telegram token loaded: {telegram_token[:8]}...")

✓ Loaded .env.local from /home/bbaron/dev/my_pet_projects/learnflow-ai/.env.local
✓ OpenAI API key loaded: sk-proj-...
✓ Telegram token loaded: 81554583...


### Bot Configuration

Все конфигурационные параметры в одном месте для упрощения настройки:

In [2]:
# Bot Configuration
TELEGRAM_TOKEN = os.getenv("TELEGRAM_TOKEN")
MODEL_NAME = "gpt-4.1-mini"
TEMPERATURE = 0.3
SOURCE_MIN_LENGTH = 50  # Минимальная длина external source для учета

print("✓ Bot configuration loaded")
print(f"  Model: {MODEL_NAME}")
print(f"  Temperature: {TEMPERATURE}")

✓ Bot configuration loaded
  Model: gpt-4.1-mini
  Temperature: 0.3


In [3]:

def resolve_api_key(key_value: str) -> str:
    """
    Resolve API key from config value.
    If starts with $, read from environment variable.
    Otherwise, use the value directly.
    """
    if key_value.startswith("$"):
        env_var = key_value[1:]  # Remove $
        value = os.getenv(env_var)
        if not value:
            raise ValueError(f"Environment variable {env_var} not found")
        return value
    return key_value

print("✓ Helper functions defined")

# Helper function to create LLM with provider config
def create_llm(model_name: str, temperature: float, api_key: str, provider_config: dict = None):
    """
    Create ChatOpenAI instance with optional custom provider config.
    
    Args:
        model_name: Model name
        temperature: Temperature
        api_key: API key
        provider_config: Provider configuration dict (with base_url, etc.)
    
    Returns:
        ChatOpenAI instance
    """
    llm_kwargs = {
        "model": model_name,
        "temperature": temperature,
        "api_key": api_key
    }
    
    # Add base_url if configured
    if provider_config and provider_config.get("base_url"):
        llm_kwargs["base_url"] = provider_config["base_url"]
        print(f"  ✓ Using custom base_url: {provider_config['base_url']}")
    
    return ChatOpenAI(**llm_kwargs)

print("✓ LLM helper function defined")

# ============================================================================
# LOAD CONFIGURATION AND INPUT DATA
# ============================================================================
import yaml
import json
from pathlib import Path

# Load M2 config
config_path = Path("M2-config.yaml")
if config_path.exists():
    with open(config_path) as f:
        m2_config = yaml.safe_load(f)
    print(f"✓ Loaded M2 configuration")
else:
    raise FileNotFoundError(f"M2-config.yaml not found at {config_path}")

# Resolve API keys from provider config
provider_config = m2_config.get("provider", {})
openai_api_key = resolve_api_key(provider_config.get("api_key", "$OPENAI_API_KEY"))

print(f"  Provider: {provider_config.get('name', 'openai')}")
print(f"  Base URL: {provider_config.get('base_url', 'default')}")
print(f"  API key loaded: {openai_api_key[:8]}...")

# Override model settings from config
MODEL_NAME = m2_config.get("model")
TEMPERATURE = m2_config.get("temperature")

print(f"  Model: {MODEL_NAME}")
print(f"  Temperature: {TEMPERATURE}")

# Load research output (previous notebook)
outputs_dir = Path(m2_config["outputs_dir"])
research_path = outputs_dir / m2_config["research_output"]

if research_path.exists():
    with open(research_path, encoding="utf-8") as f:
        research_data = json.load(f)
    
    topic = research_data["topic"]
    external_sources = research_data["final_report"]
    
    print(f"\n✓ Loaded research output from: {research_path}")
    print(f"  Topic: {topic[:60]}...")
    print(f"  Sources: {len(research_data['sources'])}")
    print(f"  Report length: {len(external_sources)} chars")
else:
    # Fallback: no research agent output
    topic = m2_config.get("topic", "Default topic")
    external_sources = ""
    print(f"\n⚠️ No research output found at {research_path}")
    print(f"  Using topic from config: {topic[:60]}...")
    print(f"  No external sources")

✓ Helper functions defined
✓ LLM helper function defined
✓ Loaded M2 configuration
  Provider: openai
  Base URL: None
  API key loaded: sk-proj-...
  Model: gpt-4.1
  Temperature: 0.3

✓ Loaded research output from: outputs/research_output.json
  Topic: Schema-guided reasoning для разработчиков LLM-агентов...
  Sources: 5
  Report length: 3247 chars


## 2. Модели данных (Pydantic)

In [4]:
class Subsection(BaseModel):
    """Document subsection"""
    title: str = Field(description="Subsection title")
    theses: List[str] = Field(
        default_factory=list,
        description="Key theses to be covered in this subsection"
    )


class Section(BaseModel):
    """Document section"""
    title: str = Field(description="Section title")
    subsections: List[Subsection] = Field(
        default_factory=list,
        description="List of subsections"
    )


class DocumentStructure(BaseModel):
    """Document structure"""
    sections: List[Section] = Field(description="List of document sections")


class NextStepDecision(BaseModel):
    """Decision: whether to revise content or finalize."""
    next_step: Literal["clarify", "finalize"] = Field(
        description=(
            "Control flow decision: "
            "'clarify' - user wants refinements, continue HITL loop; "
            "'finalize' - user approves structure, exit HITL and proceed to generation"
        )
    )

print("✓ Pydantic models defined")
print(f"  - Subsection")
print(f"  - Section")
print(f"  - DocumentStructure")
print(f"  - NextStepDecision (two-phase HITL decision model)")

✓ Pydantic models defined
  - Subsection
  - Section
  - DocumentStructure
  - NextStepDecision (two-phase HITL decision model)


## 3. Конфигурация персонализации

Встроенная конфигурация placeholder'ов для самодостаточности notebook. Значения подобраны для ML/intermediate уровня.

In [5]:
# Встроенная конфигурация placeholder'ов
PLACEHOLDER_CONFIG = {
    "subject_keywords": {
        "machine_learning": "neural networks, gradient descent, backpropagation, supervised learning, unsupervised learning, reinforcement learning, model evaluation, overfitting, underfitting, regularization, cross-validation, feature engineering, dimensionality reduction, clustering, classification, regression, decision trees, random forests, support vector machines, k-nearest neighbors, ensemble methods, deep learning, convolutional neural networks, recurrent neural networks, transformers, natural language processing, computer vision"
    },
    "role_perspective": {
        "ml_expert": "industry expert with deep practical understanding and applied knowledge"
    },
    "subject_name": {
        "machine_learning": "machine learning"
    },
    "language": {
        "russian_tech": "russian with preserved english technical terms and abbreviations"
    },
    "target_audience_inline": {
        "intermediate": "specialists with foundational understanding seeking deeper conceptual knowledge"
    },
    "target_audience_block": {
        "intermediate": "Reader has basic knowledge of core concepts and terminology. Focus on explaining mechanisms, connections between ideas, and practical applications. Assume familiarity with foundational principles."
    },
    "topic_coverage": {
        "focused": "focused on essential principles and key mechanisms"
    },
    "material_type_inline": {
        "study_material": "comprehensive study material"
    },
    "material_type_block": {
        "study_material": "Primary learning resource designed to develop conceptual understanding through structured knowledge presentation."
    },
    "explanation_depth": {
        "intermediate": "intermediate depth explaining mechanisms and causal relationships"
    },
    "style": {
        "balanced": "balanced technical precision with accessible explanations"
    }
}


def build_placeholders(
    input_content: str,
    external_sources: str = ""
) -> dict:
    """
    Построение словаря placeholder значений из встроенной конфигурации.
    
    Args:
        input_content: Тема/вопрос пользователя
        external_sources: Внешние источники (конспекты), по умолчанию пустая строка
    
    Returns:
        dict со всеми placeholder значениями для Jinja2
    """
    return {
        "subject_keywords": PLACEHOLDER_CONFIG["subject_keywords"]["machine_learning"],
        "role_perspective": PLACEHOLDER_CONFIG["role_perspective"]["ml_expert"],
        "subject_name": PLACEHOLDER_CONFIG["subject_name"]["machine_learning"],
        "language": PLACEHOLDER_CONFIG["language"]["russian_tech"],
        "target_audience_inline": PLACEHOLDER_CONFIG["target_audience_inline"]["intermediate"],
        "target_audience_block": PLACEHOLDER_CONFIG["target_audience_block"]["intermediate"],
        "topic_coverage": PLACEHOLDER_CONFIG["topic_coverage"]["focused"],
        "material_type_inline": PLACEHOLDER_CONFIG["material_type_inline"]["study_material"],
        "material_type_block": PLACEHOLDER_CONFIG["material_type_block"]["study_material"],
        "explanation_depth": PLACEHOLDER_CONFIG["explanation_depth"]["intermediate"],
        "style": PLACEHOLDER_CONFIG["style"]["balanced"],
        "input_content": input_content,
        "external_sources": external_sources
    }


print("✓ Placeholder configuration loaded")
print(f"  Total placeholders: {len(PLACEHOLDER_CONFIG)}")

✓ Placeholder configuration loaded
  Total placeholders: 11


## 4. Системные промпты

In [6]:
# Initial prompt для первоначальной генерации структуры
PLANNING_STRUCTURE_SYSTEM_PROMPT = """
KEYWORD: {{ subject_keywords }}
<!-- Keywords above activate domain expertise, use naturally if relevant -->

<role>
You are a {{ role_perspective }} specializing in {{ subject_name }}, designing document structures for {{ target_audience_inline }}.
</role>

<task>
Design a hierarchical document structure (sections with subsections and theses) that serves as a blueprint for comprehensive {{ material_type_inline }} generation.
</task>

<input_data>
  <topic>
  {{ input_content }}
  </topic>

  <external_sources>
  {{ external_sources }}
  </external_sources>
</input_data>

<structure_requirements>
  <source_integration>
    - Every concept, topic, method, or detail from <external_sources></external_sources> MUST be reflected in the structure
    - Map each element from sources to appropriate sections, subsections or theses
    - Complement sources with your domain knowledge to fill gaps and provide context
    - Ensure coherent synthesis between all available information
    - If no external sources provided, rely entirely on your domain expertise
  </source_integration>

  <hierarchy_design>
    - Design structure depth and breadth naturally fitting the topic scope and context
    - Number of sections should emerge organically (may be 1-2 for focused topics, or more for comprehensive coverage)
    - Adapt structure complexity to align with explanation depth, coverage scope, and audience needs specified below
    - Use clear, specific, self-descriptive titles
    - Maintain consistent naming style throughout
    - Avoid overly generic or vague titles
    - Each subsection represents a focused learning unit covering a specific aspect
    - Each section groups related subsections into a coherent thematic block
  </hierarchy_design>

  <content_parameters>
    <topic_coverage> {{ topic_coverage }} </topic_coverage>
    <explanation_depth> {{ explanation_depth }} </explanation_depth>
    <style> {{ style }} </style>
    <material_type> {{ material_type_block }} </material_type>
  </content_parameters>

  <target_audience> {{ target_audience_block }} </target_audience>
</structure_requirements>

<output_format>
  <language> {{ language }} </language>
</output_format>
"""

# Further prompt для HITL итераций (объединенный: decision + content)
PLANNING_STRUCTURE_FURTHER_SYSTEM_PROMPT = """
KEYWORD: {{ subject_keywords }}
<!-- Keywords above activate domain expertise, use naturally if relevant -->

<role>
You are a {{ role_perspective }} specializing in {{ subject_name }}, refining and improving document structures for {{ target_audience_inline }} based on user feedbacl.
</role>

<task>
Two-phase response process for refining document structure based on user feedback:

Phase 1 (Decision): Analyze user feedback and determine next step
  - Output: NextStepDecision schema with next_step field

Phase 2 (Content): If clarify chosen, generate refined structure
  - Output: DocumentStructure schema with updated sections
</task>

<input_data>
  <topic>
  {{ input_content }}
  </topic>

  <external_sources>
  {{ external_sources }}
  </external_sources>
</input_data>

<structure_requirements>
  <source_integration>
    - Every concept, topic, method, or detail from <external_sources></external_sources> MUST be reflected in the structure
    - Map each element from sources to appropriate sections, subsections or theses
    - Complement sources with your domain knowledge to fill gaps and provide context
    - Ensure coherent synthesis between all available information
    - If no external sources provided, rely entirely on your domain expertise
  </source_integration>

  <hierarchy_design>
    - Design structure depth and breadth naturally fitting the topic scope and context
    - Number of sections should emerge organically (may be 1-2 for focused topics, or more for comprehensive coverage)
    - Adapt structure complexity to align with explanation depth, coverage scope, and audience needs specified below
    - Use clear, specific, self-descriptive titles
    - Maintain consistent naming style throughout
    - Avoid overly generic or vague titles
    - Each subsection represents a focused learning unit covering a specific aspect
    - Each section groups related subsections into a coherent thematic block
  </hierarchy_design>

  <content_parameters>
    <topic_coverage> {{ topic_coverage }} </topic_coverage>
    <explanation_depth> {{ explanation_depth }} </explanation_depth>
    <style> {{ style }} </style>
    <material_type> {{ material_type_block }} </material_type>
  </content_parameters>

  <target_audience> {{ target_audience_block }} </target_audience>
</structure_requirements>

<refinement_requirements>
  <decision_phase>
    Analyze user feedback to determine intent:
    - If user expresses satisfaction (e.g., "всё хорошо", "отлично", "подходит", 
      "good", "perfect", "looks great", "approve") → next_step: "finalize"
    - If user requests changes, improvements, reorganization, or has questions 
      → next_step: "clarify"
  </decision_phase>

  <content_phase>
    If clarify chosen in decision phase:
    - Extract current structure from conversation history (last assistant message)
    - Apply user feedback precisely to improve structure organization
    - Maintain relevance to the original topic and external sources
    - Preserve successful aspects while addressing weak points
    - Respond constructively to structural feedback (add/remove/reorder sections)
    - Adjust hierarchy depth based on user preferences
    - Refine section and subsection titles for better clarity
    - Ensure theses alignment with user's learning objectives
    - Maintain consistency with content parameters and target audience
  </content_phase>
</refinement_requirements>

<output_format>
  <language> {{ language }} </language>
</output_format>
"""

print("✓ System prompts defined")
print(f"  - PLANNING_STRUCTURE_SYSTEM_PROMPT (initial)")
print(f"  - PLANNING_STRUCTURE_FURTHER_SYSTEM_PROMPT (two-phase HITL: decision + content)")

✓ System prompts defined
  - PLANNING_STRUCTURE_SYSTEM_PROMPT (initial)
  - PLANNING_STRUCTURE_FURTHER_SYSTEM_PROMPT (two-phase HITL: decision + content)


## 5. Функция генерации структуры

In [7]:
def generate_document_structure(
    placeholders: dict,
    model_name: str = "gpt-4.1-mini",
    temperature: float = 0.3,
    verbose: bool = True
) -> DocumentStructure:
    """
    Генерирует структуру всего документа за один вызов LLM.
    
    Args:
        placeholders: Словарь со всеми placeholder значениями
        model_name: Название модели OpenAI
        temperature: Температура генерации (0.0-1.0)
        verbose: Выводить ли промежуточную информацию
    
    Returns:
        DocumentStructure со всеми секциями
    """
    # Создаем модель с structured output
    llm = ChatOpenAI(
        model=model_name,
        temperature=temperature,
        api_key=openai_api_key
    )
    
    structured_llm = llm.with_structured_output(DocumentStructure)
    
    # Рендерим системный промпт через Jinja2
    template = Template(PLANNING_STRUCTURE_SYSTEM_PROMPT)
    rendered_prompt = template.render(**placeholders)
    
    if verbose:
        print(f"\n{'='*60}")
        print(f"Generating document structure")
        print(f"Topic: {placeholders['input_content'][:80]}{'...' if len(placeholders['input_content']) > 80 else ''}")
        print(f"Model: {model_name} (temp={temperature})")
        print(f"External sources: {'Yes' if placeholders['external_sources'] else 'No'}")
        print(f"{'='*60}\n")
    
    # Генерируем
    result = structured_llm.invoke(rendered_prompt)
    
    if verbose:
        print(f"✓ Generated structure with {len(result.sections)} sections")
        total_subsections = sum(len(s.subsections) for s in result.sections)
        print(f"✓ Total subsections: {total_subsections}")
    
    return result


print("✓ Generation function defined")

✓ Generation function defined


## 6. Telegram Bot Implementation

### 6.1. Bot Setup

In [8]:
import nest_asyncio
from aiogram import Bot, Dispatcher
from aiogram.filters import Command
from aiogram.fsm.context import FSMContext
from aiogram.fsm.state import State, StatesGroup
from aiogram.types import Message

# Применяем nest_asyncio для совместимости с Jupyter
nest_asyncio.apply()

# Создаем Bot и Dispatcher
bot = Bot(token=TELEGRAM_TOKEN)
dp = Dispatcher()

# Определяем FSM states
class StructureGeneration(StatesGroup):
    WAITING_TOPIC = State()      # Ожидание темы от пользователя
    WAITING_SOURCE = State()     # Ожидание external source
    HITL_REVIEW = State()        # HITL review и feedback loop

print("✓ Bot and Dispatcher initialized")
print("✓ FSM states defined:")
print("  - WAITING_TOPIC")
print("  - WAITING_SOURCE")
print("  - HITL_REVIEW")

✓ Bot and Dispatcher initialized
✓ FSM states defined:
  - WAITING_TOPIC
  - WAITING_SOURCE
  - HITL_REVIEW


### 6.2. Helper Function: Format Structure

In [9]:
def format_structure_for_telegram(structure: DocumentStructure) -> str:
    """
    Форматирует структуру документа для отображения в Telegram.
    
    Args:
        structure: Объект DocumentStructure
    
    Returns:
        Форматированная строка с эмодзи и индентацией
    """
    lines = []
    
    for i, section in enumerate(structure.sections, 1):
        lines.append(f"\n{i}. 📚 **{section.title}**")
        
        for j, subsection in enumerate(section.subsections, 1):
            is_last_subsection = (j == len(section.subsections))
            prefix = "└─" if is_last_subsection else "├─"
            lines.append(f"  {prefix} 📖 {subsection.title}")
            
            if subsection.theses:
                for k, thesis in enumerate(subsection.theses):
                    thesis_prefix = "  │  " if not is_last_subsection else "     "
                    lines.append(f"{thesis_prefix}• {thesis}")
    
    return "\n".join(lines)


print("✓ format_structure_for_telegram() defined")

✓ format_structure_for_telegram() defined


### 6.3. Command Handlers

In [10]:
@dp.message(Command("start"))
async def cmd_start(message: Message, state: FSMContext):
    """/start handler - начало нового сеанса генерации"""
    user_id = message.from_user.id
    print(f"\n[{user_id}] /start command")
    
    await state.clear()
    await message.answer(
        "👋 Добро пожаловать!\n\n"
        "Я помогу создать структуру образовательного документа с использованием HITL паттерна.\n\n"
        "📝 Введите тему, по которой вы хотите создать материал:"
    )
    await state.set_state(StructureGeneration.WAITING_TOPIC)


@dp.message(Command("reset"))
async def cmd_reset(message: Message, state: FSMContext):
    """/reset handler - сброс текущего состояния"""
    user_id = message.from_user.id
    print(f"\n[{user_id}] /reset command")
    
    await state.clear()
    await message.answer(
        "🔄 Состояние сброшено.\n\n"
        "📝 Введите тему, по которой вы хотите создать материал:"
    )
    await state.set_state(StructureGeneration.WAITING_TOPIC)


@dp.message(Command("help"))
async def cmd_help(message: Message):
    """/help handler - показывает инструкции"""
    user_id = message.from_user.id
    print(f"\n[{user_id}] /help command")
    
    await message.answer(
        "📖 **Инструкции по использованию**\n\n"
        "**Команды:**\n"
        "/start - Начать новый сеанс генерации\n"
        "/reset - Сбросить текущее состояние\n"
        "/help - Показать эту справку\n\n"
        "**Как это работает:**\n"
        "1. Введите тему вашего материала\n"
        "2. Опционально: добавьте конспекты или введите 'пропустить'\n"
        "3. Получите структуру документа\n"
        "4. Внесите правки или утвердите структуру\n"
        "5. Получите финальную структуру в JSON формате\n\n"
        "**Ключевые слова для утверждения:**\n"
        "да, хорошо, отлично, подходит, approve, perfect"
    )


print("✓ Command handlers registered")
print("  - /start")
print("  - /reset")
print("  - /help")

✓ Command handlers registered
  - /start
  - /reset
  - /help


### 6.4. Topic Handler

In [11]:
@dp.message(StructureGeneration.WAITING_TOPIC)
async def handle_topic(message: Message, state: FSMContext):
    """
    Handler для состояния WAITING_TOPIC.
    Сохраняет тему и переходит к запросу external source.
    """
    topic = message.text
    user_id = message.from_user.id
    
    print(f"\n[{user_id}] New session started")
    print(f"[{user_id}] Topic: {topic[:80]}{'...' if len(topic) > 80 else ''}")
    
    await state.update_data(topic=topic)
    
    await message.answer(
        f"✓ Тема сохранена: {topic}\n\n"
        f"📄 Теперь введите внешние источники (конспекты) или напишите 'пропустить':\n\n"
        f"(Для учета источника требуется минимум {SOURCE_MIN_LENGTH} символов)"
    )
    
    await state.set_state(StructureGeneration.WAITING_SOURCE)


print("✓ Topic handler registered")

✓ Topic handler registered


### 6.5. Source Handler

In [12]:
@dp.message(StructureGeneration.WAITING_SOURCE)
async def handle_source(message: Message, state: FSMContext):
    """
    Handler для состояния WAITING_SOURCE.
    Обрабатывает external source и генерирует первоначальную структуру.
    """
    data = await state.get_data()
    topic = data["topic"]
    user_id = message.from_user.id
    
    print(f"\n[{user_id}] Processing external sources")
    
    # Обработка external source
    user_input = message.text
    if len(user_input) < SOURCE_MIN_LENGTH or "пропустить" in user_input.lower():
        external_sources = ""
        print(f"[{user_id}] External sources: skipped")
        await message.answer("ℹ️ Источники не учтены (менее 50 символов или пропущено)")
    else:
        external_sources = user_input
        print(f"[{user_id}] External sources: provided ({len(external_sources)} chars)")
        await message.answer("✓ Источники учтены")
    
    # Build placeholders
    placeholders = build_placeholders(
        input_content=topic,
        external_sources=external_sources
    )
    
    # Генерация структуры
    print(f"[{user_id}] Starting initial structure generation")
    await message.answer("⏳ Генерирую структуру документа...")
    
    try:
        structure = generate_document_structure(
            placeholders=placeholders,
            model_name=MODEL_NAME,
            temperature=TEMPERATURE,
            verbose=False
        )
        
        print(f"[{user_id}] Structure generated: {len(structure.sections)} sections")
        
        # Инициализация истории сообщений
        messages = [
            {
                "role": "assistant",
                "content": structure.model_dump_json()
            }
        ]
        
        print(f"[{user_id}] Message history initialized")
        
        # Сохраняем в state
        await state.update_data(
            external_sources=external_sources,
            placeholders=placeholders,
            messages=messages  # История сообщений для HITL
        )
        
        # Форматируем и отправляем
        formatted = format_structure_for_telegram(structure)
        await message.answer(formatted)
        await message.answer(
            "\n💭 Всё устраивает?\n\n"
            "• Напишите 'да', 'отлично', 'подходит' для утверждения\n"
            "• Или опишите, что нужно изменить"
        )
        
        print(f"[{user_id}] Entering HITL review state")
        await state.set_state(StructureGeneration.HITL_REVIEW)
        
    except Exception as e:
        print(f"[{user_id}] ERROR: {str(e)}")
        await message.answer(
            f"❌ Ошибка при генерации структуры:\n{str(e)}\n\n"
            f"Попробуйте /reset для начала сначала"
        )


print("✓ Source handler registered")

✓ Source handler registered


### 6.6. HITL Feedback Handler

In [13]:
@dp.message(StructureGeneration.HITL_REVIEW)
async def handle_hitl_feedback(message: Message, state: FSMContext):
    """
    Handler для состояния HITL_REVIEW.
    Реализует двухэтапную генерацию: Decision Phase → Content Phase (условно).
    """
    data = await state.get_data()
    messages = data.get("messages", [])
    placeholders = data["placeholders"]
    user_id = message.from_user.id

    topic = placeholders["input_content"]
    external_sources = placeholders["external_sources"]
    
    print(f"\n[{user_id}] HITL feedback received: '{message.text[:50]}...'")
    
    # Добавляем user feedback в историю
    messages.append({
        "role": "user",
        "content": message.text
    })
    
    print(f"[{user_id}] Message history updated (total messages: {len(messages)})")
    
    # Phase 1: Decision
    print(f"[{user_id}] Phase 1: Starting decision analysis")
    await message.answer("🔍 Анализирую feedback...")
    
    try:
        llm = ChatOpenAI(
            model=MODEL_NAME,
            temperature=TEMPERATURE,
            api_key=openai_api_key
        )
        
        # Decision LLM с NextStepDecision schema
        llm_decision = llm.with_structured_output(NextStepDecision)
        
        # Рендерим system prompt
        template = Template(PLANNING_STRUCTURE_FURTHER_SYSTEM_PROMPT)
        rendered_system = template.render(**placeholders)
        
        # Формируем сообщения для LLM: system + history
        llm_messages = [
            {"role": "system", "content": rendered_system}
        ] + messages
        
        # Phase 1: Decision call
        decision = llm_decision.invoke(llm_messages)
        
        print(f"[{user_id}] Phase 1 complete: decision = {decision.next_step}")
        
        # Добавляем decision в историю
        messages.append({
            "role": "assistant",
            "content": decision.model_dump_json()
        })
        
        if decision.next_step == "finalize":
            print(f"[{user_id}] Finalizing structure (no Phase 2)")
            
            # Финализация - извлекаем последнюю структуру из истории
            # Последняя структура - это предпоследнее assistant сообщение (до decision)
            structure_messages = [
                msg for msg in messages 
                if msg["role"] == "assistant" and "sections" in msg["content"]
            ]
            
            if structure_messages:
                final_structure_json = structure_messages[-1]["content"]
                
                print(f"[{user_id}] Structure approved and finalized")

                # Save planning output for next notebook (M2-02)
                output_data = {
                    "topic": topic,
                    "external_sources": external_sources,
                    "structure": json.loads(final_structure_json)
                }

                outputs_dir.mkdir(exist_ok=True)
                planning_output_path = outputs_dir / m2_config["planning_output"]
                with open(planning_output_path, "w", encoding="utf-8") as f:
                    json.dump(output_data, f, indent=2, ensure_ascii=False)

                print(f"[{user_id}] Saved planning output to: {planning_output_path}")

                await message.answer(
                    "✅ **Структура утверждена!**\n\n"
                    "📋 Финальная структура (JSON):"
                )
                
                # Отправляем JSON
                if len(final_structure_json) > 4000:
                    chunks = [final_structure_json[i:i+4000] for i in range(0, len(final_structure_json), 4000)]
                    for chunk in chunks:
                        await message.answer(f"```json\n{chunk}\n```")
                else:
                    await message.answer(f"```json\n{final_structure_json}\n```")
                
                await message.answer(
                    "\n🎉 Готово! Используйте /start для создания новой структуры"
                )
                
                print(f"[{user_id}] Session completed, clearing state")
                await state.clear()
                await state.set_state(StructureGeneration.WAITING_TOPIC)
            else:
                print(f"[{user_id}] ERROR: No structure found in message history")
                await message.answer("❌ Ошибка: не найдена структура в истории")
                
        else:  # decision.next_step == "clarify"
            # Phase 2: Content generation
            print(f"[{user_id}] Phase 2: Starting content regeneration")
            await message.answer("⏳ Перегенерирую структуру...")
            
            # Content LLM с DocumentStructure schema
            llm_content = llm.with_structured_output(DocumentStructure)
            
            # Используем ту же историю + decision
            updated_structure = llm_content.invoke(llm_messages)
            
            print(f"[{user_id}] Phase 2 complete: {len(updated_structure.sections)} sections")
            
            # Добавляем обновленную структуру в историю
            messages.append({
                "role": "assistant",
                "content": updated_structure.model_dump_json()
            })
            
            # Сохраняем обновленную историю
            await state.update_data(messages=messages)
            
            print(f"[{user_id}] Message history updated (total messages: {len(messages)})")
            
            # Отправляем обновленную структуру
            formatted = format_structure_for_telegram(updated_structure)
            await message.answer(formatted)
            
            await message.answer(
                "\n💭 Теперь устраивает?\n\n"
                "• Напишите 'да', 'отлично', 'подходит' для утверждения\n"
                "• Или опишите, что еще нужно изменить"
            )
            
            print(f"[{user_id}] Waiting for next feedback")
    
    except Exception as e:
        print(f"[{user_id}] ERROR: {str(e)}")
        await message.answer(
            f"❌ Ошибка при обработке feedback:\n{str(e)}\n\n"
            f"Попробуйте /reset для начала сначала"
        )


print("✓ HITL feedback handler registered (two-phase: decision → content)")

✓ HITL feedback handler registered (two-phase: decision → content)


## 7. Run Bot

### Инструкции:

1. Запустите эту cell
2. Откройте Telegram и найдите вашего бота
3. Отправьте `/start` для начала работы
4. Следуйте инструкциям бота

**Для остановки бота:**
- Нажмите Ctrl+C в терминале Jupyter
- Или прервите выполнение kernel (Kernel → Interrupt)

### Пример сценария:

```
User: /start
Bot: Введите тему...

User: Векторные базы данных: Qdrant для production
Bot: Введите источники или 'пропустить'...

User: пропустить
Bot: [показывает структуру] Всё устраивает?

User: добавь больше про scaling и performance
Bot: [показывает обновленную структуру] Теперь устраивает?

User: отлично
Bot: [показывает финальную JSON структуру]
```

## Выводы

Этот notebook демонстрирует практическую реализацию принципов декомпозиции генерации, которые независимо подтверждают выводы исследования "Navigating the Path of Writing".

### ✅ Достигнутые результаты:

1. **Предсказуемость** - структурированный подход делает результат управляемым
2. **Эффективность** - экономия 15x по токенам при итерациях (600 vs 9000)
3. **Качество** - разделение ролей "архитектор" и "строитель" улучшает фокус
4. **Интеграция** - естественное включение внешних источников (конспектов) в планирование

### 🚀 Применение в LearnFlow AI:

Эта реализация станет основой для миграции с текущей двойной генерации на эффективную декомпозированную архитектуру, где:
- Конспекты учитываются с самого начала
- Структура валидируется до генерации контента
- Секции генерируются параллельно для скорости
- HITL взаимодействие становится естественным и дешевым

### 📚 Дополнительные материалы:

- [LearnFlow AI Repository](https://github.com/Bbar0n234/learnflow-ai)
- [Navigating the Path of Writing (arXiv)](https://www.arxiv.org/abs/2404.13919)

## 8. Демонстрация: Полный цикл генерации

### Реальный пример workflow

Рассмотрим, как происходит весь процесс на примере генерации материала по векторным БД:

```
👤 User: /start
🤖 Bot: 👋 Добро пожаловать!
       Я помогу создать структуру образовательного документа...
       📝 Введите тему, по которой вы хотите создать материал:

👤 User: Векторная база Qdrant: исчерпывающее руководство
🤖 Bot: ✓ Тема сохранена: Векторная база Qdrant: исчерпывающее руководство
       📄 Теперь введите внешние источники (конспекты) или напишите 'пропустить':

👤 User: [Вставляет конспект с заметками о методах индексирования HNSW, примерами 
        кода на Python, сравнением с Pinecone - 1077 символов]
🤖 Bot: ✓ Источники учтены
       ⏳ Генерирую структуру документа...
       
       1. 📚 **Основы векторных баз данных**
          ├─ 📖 Что такое векторные БД и зачем они нужны
          ├─ 📖 Отличия от традиционных реляционных БД
          └─ 📖 Области применения в ML и NLP
       
       2. 📚 **Архитектура и алгоритмы Qdrant**
          ├─ 📖 Компоненты системы и их взаимодействие
          ├─ 📖 Алгоритм индексирования HNSW
          ├─ 📖 Хранение данных и персистентность
          └─ 📖 Сравнение с Pinecone (из конспекта)
       
       3. 📚 **Практическое использование**
          ├─ 📖 Установка и первоначальная настройка
          ├─ 📖 Python SDK и примеры кода (из конспекта)
          └─ 📖 Оптимизация поисковых запросов
          
       💭 Всё устраивает?
       • Напишите 'да', 'отлично', 'подходит' для утверждения
       • Или опишите, что нужно изменить

👤 User: Давай более сжато, не хочу столько много материала
🤖 Bot: 🔍 Анализирую feedback...
       ⏳ Перегенерирую структуру...
       
       1. 📚 **Введение в Qdrant**
          ├─ 📖 Основные концепции векторного поиска
          └─ 📖 Преимущества Qdrant над альтернативами
       
       2. 📚 **Техническая реализация**
          ├─ 📖 HNSW индексирование (из конспекта)
          └─ 📖 API и Python примеры (из конспекта)
       
       3. 📚 **Практическое применение**
          └─ 📖 Best practices и оптимизация
          
       💭 Теперь устраивает?

👤 User: Это слишком сжато, нужно что-то среднее
🤖 Bot: 🔍 Анализирую feedback...
       ⏳ Перегенерирую структуру...
       
       1. 📚 **Основы Qdrant**
          ├─ 📖 Векторные базы данных: концепция и применение
          ├─ 📖 Архитектура Qdrant
          └─ 📖 Сравнение с Pinecone (из конспекта)
       
       2. 📚 **Алгоритмы и индексирование**
          ├─ 📖 HNSW: принципы работы (из конспекта)
          └─ 📖 Оптимизация производительности
       
       3. 📚 **Работа с Qdrant**
          ├─ 📖 Python SDK и примеры (из конспекта)
          └─ 📖 Deployment и масштабирование
       
       4. 📚 **Best Practices**
          ├─ 📖 Выбор параметров индексирования
          └─ 📖 Мониторинг и отладка
          
       💭 Теперь устраивает?

👤 User: Всё чётко, давай дальше
🤖 Bot: ✅ **Структура утверждена!**
       
       📋 Финальная структура (JSON):
       ```json
       {
         "sections": [
           {
             "title": "Основы Qdrant",
             "subsections": [
               {"title": "Векторные базы данных: концепция и применение", "theses": [...]},
               {"title": "Архитектура Qdrant", "theses": [...]},
               {"title": "Сравнение с Pinecone", "theses": [...]}
             ]
           },
           ...
         ]
       }
       ```
       
       🎉 Готово! Используйте /start для создания новой структуры
```

### 💡 Ключевые моменты демонстрации:

1. **Учет конспектов** - система явно показывает, какие части структуры основаны на предоставленных материалах (пометка "из конспекта")

2. **Итеративное улучшение** - три итерации заняли ~15 секунд и ~600 токенов суммарно
   - Итерация 1: слишком подробно → упростить
   - Итерация 2: слишком сжато → добавить детали  
   - Итерация 3: оптимально → утвердить

3. **Альтернатива без декомпозиции** - перегенерация полного документа 3 раза заняла бы:
   - ~90 секунд (vs 15 секунд)
   - ~9000 токенов (vs 600 токенов)
   - Без гарантии улучшения структуры

4. **Контроль пользователя** - пользователь видит и корректирует структуру ДО генерации контента, избегая траты ресурсов на неподходящий материал

5. **Автоматизация решений** - система автоматически определяет намерение пользователя:
   - "давай дальше", "всё чётко" → финализация
   - конкретные правки → новая итерация

In [14]:
print("🤖 Запуск Telegram бота...")
print("\nБот готов к работе! Откройте Telegram и отправьте /start")
print("\nДля остановки: Ctrl+C или прервите kernel\n")
print("="*60)

try:
    await dp.start_polling(bot)
except KeyboardInterrupt:
    print("\n" + "="*60)
    print("✓ Бот остановлен")
finally:
    await bot.session.close()
    print("✓ Сессия закрыта")

🤖 Запуск Telegram бота...

Бот готов к работе! Откройте Telegram и отправьте /start

Для остановки: Ctrl+C или прервите kernel


[979557959] /start command

[979557959] New session started
[979557959] Topic: Schema-guided reasoning для разработчиков LLM-агентов

[979557959] Processing external sources
[979557959] External sources: provided (3225 chars)
[979557959] Starting initial structure generation
[979557959] Structure generated: 6 sections
[979557959] Message history initialized
[979557959] Entering HITL review state

[979557959] HITL feedback received: 'Давай более сжато...'
[979557959] Message history updated (total messages: 2)
[979557959] Phase 1: Starting decision analysis
[979557959] Phase 1 complete: decision = clarify
[979557959] Phase 2: Starting content regeneration
[979557959] Phase 2 complete: 3 sections
[979557959] Message history updated (total messages: 4)
[979557959] Waiting for next feedback

[979557959] HITL feedback received: 'Принято!...'
[979557959] Message hi

Received SIGINT signal


✓ Сессия закрыта
